# Matched experiment v3 — the lab (one notebook per Kaggle account)

Runs any set of victims through **every** attack: the paper's (Original MIA, + Variance, Fluctuation, + Variance),
the v2 fixed AFMIA, and the new lab attacks — plus FGSM evasion and the paper's metrics.

1. **Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. Add the input **NIH Chest X-rays** (`nih-chest-xrays/data`).
2. For `IMPORT_RUNS`: also attach the old outputs that contain those victims (an old notebook output, or a
   private Kaggle *dataset* made from your local `results 3` / `results 5` / … folders).
3. Set the variables below, then **Save Version → Save & Run All (Commit)**.
4. If `STATUS.md` says *NOT FINISHED*, add **this notebook's latest output** as an input and Save Version again.

Full instructions: `Matched_Experiments_v3/README.md` and `Matched_Experiments_v3/kaggle/README_KAGGLE.md`.


In [ ]:
# ===================== EDIT (one preset per account — see README_KAGGLE.md) =====================
LABEL       = "attack-only"   # any name for this account's run
VICTIMS     = ""              # victims to TRAIN, e.g. "fgsm_rs:4:0.5 fgsm_rs:8:0.5"  ("std" = standard)
IMPORT_RUNS = "std_noreg_seed42 fgsm_noreg_seed42"   # finished victims to reuse from attached old outputs
SEEDS       = "42"            # seeds of the TRAINED victims
SMOKE       = False           # True = quick pipeline test (writes results_v3_smoke/)
# Presets:  account 1 -> VICTIMS = ""                           IMPORT_RUNS = "std_noreg_seed42 fgsm_noreg_seed42"
#           account 2 -> VICTIMS = "fgsm_rs:4:0.5 fgsm_rs:8:0.5" IMPORT_RUNS = ""
#           account 3 -> VICTIMS = "fgsm_rs:4:0 pgd3:4:0.5"      IMPORT_RUNS = ""      (optional)
# ===============================================================================================
REPO   = "https://github.com/NCRL-IIITB/Evasion_MIA.git"
BRANCH = "main"
CODE   = "/tmp/Evasion_MIA"     # code lives outside /kaggle/working (not saved as output)


In [ ]:
# Get the code (git clone, or git pull if it is already there)
import os, subprocess

url = REPO
try:   # private repository: add a Kaggle secret called GITHUB_TOKEN (Add-ons -> Secrets)
    from kaggle_secrets import UserSecretsClient
    url = REPO.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
except Exception:
    pass

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:   # never print the URL: it may contain the token
        raise RuntimeError("git " + args[0] + " failed:\n" + r.stderr.replace(url, REPO))
    return r.stdout

if os.path.isdir(os.path.join(CODE, ".git")):
    git("-C", CODE, "pull", "--ff-only")
else:
    git("clone", "--depth", "1", "--branch", BRANCH, url, CODE)
print("Code version:", git("-C", CODE, "log", "-1", "--format=%h  %s  (%ci)"))
for d in ("Matched_Experiments", "Matched_Experiments_v2", "Matched_Experiments_v3"):
    assert os.path.isdir(f"{CODE}/{d}"), f"{d} is not in the pushed code (v3 needs all three folders)"


In [ ]:
# Run the lab: restores the previous output (if attached), imports / trains victims, attacks.
# It stops by itself before Kaggle's 12-hour limit so the output is always saved.
cmd = ["python", "-u", f"{CODE}/Matched_Experiments_v3/kaggle/kaggle_pipeline.py",
       "--label", LABEL, "--seeds", *SEEDS.split()]
if VICTIMS.split():
    cmd += ["--victims", *VICTIMS.split()]
if IMPORT_RUNS.split():
    cmd += ["--import_runs", *IMPORT_RUNS.split()]
if SMOKE:
    cmd += ["--smoke"]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("pipeline exit code:", proc.returncode)


In [ ]:
# Status of this account (also saved as results_v3/STATUS.md in the output)
res = "/kaggle/working/" + ("results_v3_smoke" if SMOKE else "results_v3")
print(open(f"{res}/STATUS.md").read() if os.path.exists(f"{res}/STATUS.md") else "no STATUS.md yet")
